In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch6. RNN기반의 Seq2Seq(스마트 번역기)</font>**
- Google Neural Machine Translation(GNMT)
- RNN기반의 Seq2Seq방식
- 인코더입력/디코더입력(모범답안) - 디코더 출력(답안) ; 인코더와 디코더가 연결된 구조

# 1. 패키지 import 및 하이퍼 파라미터

In [2]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# 하이퍼 파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 학습데이터

In [7]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor = raw.values.tolist() # 데이터프레임을 list로 변환
print(eng_kor[:3])
print('학습할 영-한 데이터 갯수 :', len(eng_kor))

[['cold', '감기'], ['come', '오다'], ['cook', '요리']]
학습할 영-한 데이터 갯수 : 110


In [14]:
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
korean = ''.join([data[1] for data in eng_kor])
print(set([ch for ch in korean]))

{'적', '얼', '류', '개', '익', '놀', '입', '이', '유', '휴', '동', '수', '노', '비', '나', '들', '인', '운', '계', '그', '칙', '매', '약', '규', '책', '금', '시', '키', '랑', '램', '메', '을', '다', '농', '광', '회', '남', '장', '팔', '관', '손', '어', '움', '깊', '굴', '사', '색', '흐', '망', '리', '머', '좋', '우', '먼', '무', '부', '음', '피', '출', '녀', '고', '작', '요', '복', '모', '각', '상', '위', '택', '게', '쪽', '읽', '방', '바', '거', '구', '감', '한', '것', '획', '날', '크', '제', '식', '험', '의', '통', '생', '소', '도', '름', '서', '반', '릎', '멍', '합', '간', '연', '실', '탈', '늦', '뉴', '편', '자', '분', '목', '물', '언', '명', '단', '오', '짜', '스', '은', '람', '미', '여', '싸', '넓', '얇', '핑', '뿌', '옥', '붕', '래', '용', '내', '높', '왼', '번', '해', '파', '아', '기', '가', '찾', '지', '많', '선', '행', '쉽', '주'}
